In [1]:
from pyspark.sql.functions import col, trim, upper, regexp_replace, to_date, when, lit, current_timestamp, length
from pyspark.sql.types import LongType, DoubleType 

SILVER_TABLE = "silver_ae_monthly"
REJECTED_TABLE = "silver_ae_rejected_rows"

print("Config set")

StatementMeta(, 282ff829-aaf1-4e02-86f8-9cfee9472f4b, 3, Finished, Available, Finished, False)

Config set


In [2]:
df_bronze = spark.read.table("bronze_ae_monthly")

print(f"Rows read from Bronze: {df_bronze.count()}")
df_bronze.printSchema()

StatementMeta(, 282ff829-aaf1-4e02-86f8-9cfee9472f4b, 4, Finished, Available, Finished, False)

Rows read from Bronze: 7405
root
 |-- period: string (nullable = true)
 |-- org_code: string (nullable = true)
 |-- parent_org: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- ae_attendances_type1: string (nullable = true)
 |-- ae_attendances_type2: string (nullable = true)
 |-- ae_attendances_other: string (nullable = true)
 |-- ae_attendances_booked_type1: string (nullable = true)
 |-- ae_attendances_booked_type2: string (nullable = true)
 |-- ae_attendances_booked_other: string (nullable = true)
 |-- attendance_over_4hrs_type1: string (nullable = true)
 |-- attendance_over_4hrs_type2: string (nullable = true)
 |-- attendance_over_4hrs_other: string (nullable = true)
 |-- attendance_over_4hrs_booked_type1: string (nullable = true)
 |-- attendance_over_4hrs_booked_type2: string (nullable = true)
 |-- attendance_over_4hrs_booked_other: string (nullable = true)
 |-- patient_waited_4_12hrs_dta: string (nullable = true)
 |-- patient_waited_12plus_hrs_dta: string (nul

In [17]:
df_cleaned = (
    df_bronze
    # clean text columns
    .withColumn("org_code", trim(upper(col("org_code"))))
    .withColumn("parent_org", trim(upper(col("parent_org"))))
    .withColumn("org_name", trim(col("org_name")))

    # strip MSitAE- prefix then parse period to a proper date — NHS format is "April 2021"
    .withColumn("period", regexp_replace(col("period"), "MSitAE-", ""))
    .withColumn("period_date", to_date(col("period"), "MMMM-yyyy"))

    # replace suppressed values (-, *, —) with null before casting to numbers
    .withColumn("ae_attendances_type1", 
        when(col("ae_attendances_type1").rlike("^[0-9]+$"), col("ae_attendances_type1"))
        .otherwise(lit(None)))
    .withColumn("ae_attendances_type2",
        when(col("ae_attendances_type2").rlike("^[0-9]+$"), col("ae_attendances_type2"))
        .otherwise(lit(None)))
    .withColumn("ae_attendances_other",
        when(col("ae_attendances_other").rlike("^[0-9]+$"), col("ae_attendances_other"))
        .otherwise(lit(None)))
    .withColumn("attendance_over_4hrs_type1",
        when(col("attendance_over_4hrs_type1").rlike("^[0-9]+$"), col("attendance_over_4hrs_type1"))
        .otherwise(lit(None)))
    .withColumn("attendance_over_4hrs_type2",
        when(col("attendance_over_4hrs_type2").rlike("^[0-9]+$"), col("attendance_over_4hrs_type2"))
        .otherwise(lit(None)))
    .withColumn("attendance_over_4hrs_other",
        when(col("attendance_over_4hrs_other").rlike("^[0-9]+$"), col("attendance_over_4hrs_other"))
        .otherwise(lit(None)))
    .withColumn("ae_attendances_booked_type1",
        when(col("ae_attendances_booked_type1").rlike("^[0-9]+$"), col("ae_attendances_booked_type1"))
        .otherwise(lit(None)))
    .withColumn("ae_attendances_booked_type2",
        when(col("ae_attendances_booked_type2").rlike("^[0-9]+$"), col("ae_attendances_booked_type2"))
        .otherwise(lit(None)))
    .withColumn("ae_attendances_booked_other",
        when(col("ae_attendances_booked_other").rlike("^[0-9]+$"), col("ae_attendances_booked_other"))
        .otherwise(lit(None)))
    .withColumn("attendance_over_4hrs_booked_type1",
        when(col("attendance_over_4hrs_booked_type1").rlike("^[0-9]+$"), col("attendance_over_4hrs_booked_type1"))
        .otherwise(lit(None)))
    .withColumn("attendance_over_4hrs_booked_type2",
        when(col("attendance_over_4hrs_booked_type2").rlike("^[0-9]+$"), col("attendance_over_4hrs_booked_type2"))
        .otherwise(lit(None)))
    .withColumn("attendance_over_4hrs_booked_other",
        when(col("attendance_over_4hrs_booked_other").rlike("^[0-9]+$"), col("attendance_over_4hrs_booked_other"))
        .otherwise(lit(None)))
    .withColumn("patient_waited_4_12hrs_dta",
        when(col("patient_waited_4_12hrs_dta").rlike("^[0-9]+$"), col("patient_waited_4_12hrs_dta"))
        .otherwise(lit(None)))
    .withColumn("patient_waited_12plus_hrs_dta",
        when(col("patient_waited_12plus_hrs_dta").rlike("^[0-9]+$"), col("patient_waited_12plus_hrs_dta"))
        .otherwise(lit(None)))
    .withColumn("emergency_admissions_type1",
        when(col("emergency_admissions_type1").rlike("^[0-9]+$"), col("emergency_admissions_type1"))
        .otherwise(lit(None)))
    .withColumn("emergency_admissions_type2",
        when(col("emergency_admissions_type2").rlike("^[0-9]+$"), col("emergency_admissions_type2"))
        .otherwise(lit(None)))
    .withColumn("emergency_admissions_other",
        when(col("emergency_admissions_other").rlike("^[0-9]+$"), col("emergency_admissions_other"))
        .otherwise(lit(None)))
    .withColumn("other_emergency_admissions",
        when(col("other_emergency_admissions").rlike("^[0-9]+$"), col("other_emergency_admissions"))
        .otherwise(lit(None)))

    # cast all numeric columns from string to long
    .withColumn("ae_attendances_type1", col("ae_attendances_type1").cast(LongType()))
    .withColumn("ae_attendances_type2", col("ae_attendances_type2").cast(LongType()))
    .withColumn("ae_attendances_other", col("ae_attendances_other").cast(LongType()))
    .withColumn("ae_attendances_booked_type1", col("ae_attendances_booked_type1").cast(LongType()))
    .withColumn("ae_attendances_booked_type2", col("ae_attendances_booked_type2").cast(LongType()))
    .withColumn("ae_attendances_booked_other", col("ae_attendances_booked_other").cast(LongType()))
    .withColumn("attendance_over_4hrs_type1", col("attendance_over_4hrs_type1").cast(LongType()))
    .withColumn("attendance_over_4hrs_type2", col("attendance_over_4hrs_type2").cast(LongType()))
    .withColumn("attendance_over_4hrs_other", col("attendance_over_4hrs_other").cast(LongType()))
    .withColumn("attendance_over_4hrs_booked_type1", col("attendance_over_4hrs_booked_type1").cast(LongType()))
    .withColumn("attendance_over_4hrs_booked_type2", col("attendance_over_4hrs_booked_type2").cast(LongType()))
    .withColumn("attendance_over_4hrs_booked_other", col("attendance_over_4hrs_booked_other").cast(LongType()))
    .withColumn("patient_waited_4_12hrs_dta", col("patient_waited_4_12hrs_dta").cast(LongType()))
    .withColumn("patient_waited_12plus_hrs_dta", col("patient_waited_12plus_hrs_dta").cast(LongType()))
    .withColumn("emergency_admissions_type1", col("emergency_admissions_type1").cast(LongType()))
    .withColumn("emergency_admissions_type2", col("emergency_admissions_type2").cast(LongType()))
    .withColumn("emergency_admissions_other", col("emergency_admissions_other").cast(LongType()))
    .withColumn("other_emergency_admissions", col("other_emergency_admissions").cast(LongType()))

    # add silver audit column
    .withColumn("silver_processed_at", current_timestamp())
)

print(f"Rows after cleaning: {df_cleaned.count()}")
print(f"Column count in silver: {len(df_cleaned.columns)}")
df_cleaned.printSchema()

StatementMeta(, 282ff829-aaf1-4e02-86f8-9cfee9472f4b, 19, Finished, Available, Finished, False)

Rows after cleaning: 7405
Column count in silver: 26
root
 |-- period: string (nullable = true)
 |-- org_code: string (nullable = true)
 |-- parent_org: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- ae_attendances_type1: long (nullable = true)
 |-- ae_attendances_type2: long (nullable = true)
 |-- ae_attendances_other: long (nullable = true)
 |-- ae_attendances_booked_type1: long (nullable = true)
 |-- ae_attendances_booked_type2: long (nullable = true)
 |-- ae_attendances_booked_other: long (nullable = true)
 |-- attendance_over_4hrs_type1: long (nullable = true)
 |-- attendance_over_4hrs_type2: long (nullable = true)
 |-- attendance_over_4hrs_other: long (nullable = true)
 |-- attendance_over_4hrs_booked_type1: long (nullable = true)
 |-- attendance_over_4hrs_booked_type2: long (nullable = true)
 |-- attendance_over_4hrs_booked_other: long (nullable = true)
 |-- patient_waited_4_12hrs_dta: long (nullable = true)
 |-- patient_waited_12plus_hrs_dta: long (nullab

In [18]:
# Valid rows — must have org_code, period_date, and at least type1 attendance
df_valid = df_cleaned.filter(
    col("org_code").isNotNull() &
    col("period_date").isNotNull() &
    col("org_code").rlike("^[A-Z0-9]+$") & 
    ~upper(col("period")).contains("TOTAL")
)

# Rejected rows — anything that failed validation
df_rejected = df_cleaned.filter(
    col("org_code").isNull() |
    col("period_date").isNull() |
    ~col("org_code").rlike("^[A-Z0-9]+$") |
    upper(col("period")).contains("TOTAL")

)

print(f"Valid rows:    {df_valid.count()}")
print(f"Rejected rows: {df_rejected.count()}")

StatementMeta(, 282ff829-aaf1-4e02-86f8-9cfee9472f4b, 20, Finished, Available, Finished, False)

Valid rows:    7369
Rejected rows: 36


In [25]:
# Write valid rows to Silver
(
    df_valid.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# df_valid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)

# Write rejected rows for auditability
(
    df_rejected.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJECTED_TABLE)
)

print(f"✅ Silver table written: {spark.read.table(SILVER_TABLE).count()} rows")
print(f"✅ Rejected table written: {spark.read.table(REJECTED_TABLE).count()} rows")

StatementMeta(, 282ff829-aaf1-4e02-86f8-9cfee9472f4b, 27, Finished, Available, Finished, True)

✅ Silver table written: 7369 rows
✅ Rejected table written: 36 rows
